# Retail Banking Customer Analytics — Term Deposit Prediction


In [35]:
import sys
!{sys.executable} -m pip install xgboost

  Using cached xgboost-3.4.1-py3-none-win_amd64.whl.metadata (2.0 kB)
Using cached xgboost-3.4.1-py3-none-win_amd64.whl (48.9 MB)



[notice] A new release of pip is available: 25.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [36]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score, classification_report, confusion_matrix
from xgboost import XGBClassifier

In [37]:
file_path=r"C:\Users\Mikey\Desktop\Project\Project_file_1.csv"
df = pd.read_csv(file_path)
print(df.shape)
df.head()

(45211, 17)


,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y
0,58,management,married,tertiary,no,2143,yes,no,unknown,5,may,261,1,-1,0,unknown,no
1,44,technician,single,secondary,no,29,yes,no,unknown,5,may,151,1,-1,0,unknown,no
2,33,entrepreneur,married,secondary,no,2,yes,yes,unknown,5,may,76,1,-1,0,unknown,no
3,47,blue-collar,married,unknown,no,1506,yes,no,unknown,5,may,92,1,-1,0,unknown,no
4,33,unknown,single,unknown,no,1,no,no,unknown,5,may,198,1,-1,0,unknown,no


In [38]:
df.dtypes

age           int64
job          object
marital      object
education    object
default      object
balance       int64
housing      object
loan         object
contact      object
day           int64
month        object
duration      int64
campaign      int64
pdays         int64
previous      int64
poutcome     object
y            object
dtype: object

In [39]:
df.isna().sum()

age          0
job          0
marital      0
education    0
default      0
balance      0
housing      0
loan         0
contact      0
day          0
month        0
duration     0
campaign     0
pdays        0
previous     0
poutcome     0
y            0
dtype: int64

In [40]:
df["y"].value_counts()

y
no     39922
yes     5289
Name: count, dtype: int64

## Clean and engineer features

Decisions:
- Drop `duration`: only known after a call happens, leaks the outcome.
- Drop `poutcome`: excluded per project decision.
- Replace `pdays` with a simple flag: were they contacted before or not.
  (`pdays = -1` means never contacted; any value `>= 0` means they were.)

In [41]:
# convert target to 0/1
df["y"] = df["y"].map({"yes": 1, "no": 0})

In [42]:
# drop leaky / excluded columns
df = df.drop(columns=["duration", "poutcome"])

In [43]:
# was_previously_contacted flag, replacing pdays
df["was_previously_contacted"] = (df["pdays"] >= 0).astype(int)
df = df.drop(columns=["pdays"])

df["was_previously_contacted"].value_counts()

was_previously_contacted
0    36954
1     8257
Name: count, dtype: int64

In [ ]:
df.groupby("was_previously_contacted")["y"].mean()

was_previously_contacted
0    0.091573
1    0.230713
Name: y, dtype: float64

In [45]:
df.columns.tolist()

['age',
 'job',
 'marital',
 'education',
 'default',
 'balance',
 'housing',
 'loan',
 'contact',
 'day',
 'month',
 'campaign',
 'previous',
 'y',
 'was_previously_contacted']

## Categorical columns

In [ ]:
categorical_cols = ["job", "marital", "education", "default", "housing", "loan", "contact", "month"]

X = df.drop(columns=["y"])
y = df["y"]

X = pd.get_dummies(X, columns=categorical_cols, drop_first=True)
X.shape

(45211, 38)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

print("Train:", X_train.shape, "Test:", X_test.shape)
print("Train positive rate:", y_train.mean().round(3))
print("Test positive rate:", y_test.mean().round(3))

Train: (36168, 38) Test: (9043, 38)
Train positive rate: 0.117
Test positive rate: 0.117


## Model 1 — Logistic Regression

Numeric columns are scaled first since Logistic Regression is sensitive to scale.
`class_weight="balanced"` helps because only ~11.7% of customers say yes.

In [48]:
numeric_cols = ["age", "balance", "day", "campaign", "previous", "was_previously_contacted"]

scaler = StandardScaler()
X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()
X_train_scaled[numeric_cols] = scaler.fit_transform(X_train[numeric_cols])
X_test_scaled[numeric_cols] = scaler.transform(X_test[numeric_cols])

In [49]:
logreg = LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42)
logreg.fit(X_train_scaled, y_train)

logreg_proba = logreg.predict_proba(X_test_scaled)[:, 1]

In [50]:
print("ROC-AUC:", round(roc_auc_score(y_test, logreg_proba), 4))
print("PR-AUC:", round(average_precision_score(y_test, logreg_proba), 4))

ROC-AUC: 0.7625
PR-AUC: 0.3439


In [51]:
logreg_pred = (logreg_proba >= 0.5).astype(int)
print(classification_report(y_test, logreg_pred, target_names=["no", "yes"]))

              precision    recall  f1-score   support

          no       0.94      0.71      0.81      7985
         yes       0.24      0.68      0.35      1058

    accuracy                           0.71      9043
   macro avg       0.59      0.70      0.58      9043
weighted avg       0.86      0.71      0.76      9043



In [52]:
confusion_matrix(y_test, logreg_pred)

array([[5689, 2296],
       [ 335,  723]])

## Model 2 — XGBoost

`scale_pos_weight` also helps balance the rare "yes" class, similar idea to
`class_weight="balanced"` above but for XGBoost.

In [53]:
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
print("scale_pos_weight:", round(scale_pos_weight, 2))

scale_pos_weight: 7.55


In [54]:
!pip install xgboost

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: C:\Python314\python.exe -m pip install --upgrade pip


In [55]:
xgb = XGBClassifier(
    n_estimators=300,
    max_depth=4,
    learning_rate=0.05,
    scale_pos_weight=scale_pos_weight,
    eval_metric="logloss",
    random_state=42
)
xgb.fit(X_train, y_train)

xgb_proba = xgb.predict_proba(X_test)[:, 1]

In [56]:
print("ROC-AUC:", round(roc_auc_score(y_test, xgb_proba), 4))
print("PR-AUC:", round(average_precision_score(y_test, xgb_proba), 4))

ROC-AUC: 0.7984
PR-AUC: 0.4061


In [57]:
xgb_pred = (xgb_proba >= 0.5).astype(int)
print(classification_report(y_test, xgb_pred, target_names=["no", "yes"]))

              precision    recall  f1-score   support

          no       0.95      0.82      0.88      7985
         yes       0.33      0.65      0.43      1058

    accuracy                           0.80      9043
   macro avg       0.64      0.74      0.66      9043
weighted avg       0.87      0.80      0.83      9043



In [58]:
confusion_matrix(y_test, xgb_pred)

array([[6569, 1416],
       [ 372,  686]])

## Comparing the two models

XGBoost should score higher on ROC-AUC and PR-AUC since it can capture
non-linear patterns that Logistic Regression can't.

In [ ]:
comparison = pd.DataFrame({
    "model": ["Logistic Regression", "XGBoost"],
    "roc_auc": [roc_auc_score(y_test, logreg_proba), roc_auc_score(y_test, xgb_proba)],
    "pr_auc": [average_precision_score(y_test, logreg_proba), average_precision_score(y_test, xgb_proba)]})
comparison

,model,roc_auc,pr_auc
0,Logistic Regression,0.762508,0.343925
1,XGBoost,0.798375,0.406065
